In [1]:
!pip install -q segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.6/154.6 kB 7.0 MB/s eta 0:00:00


In [2]:
print("hello")

hello


In [3]:
import os
import glob
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
import segmentation_models_pytorch as smp
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
np.random.seed(42)

defects = ["defect_a", "defect_b", "defect_c", "defect_d"]
HEIGHT, WIDTH = 128, 800
BATCH_SIZE = 16
EPOCHS = 25               # per model (3 models are trained)
PATIENCE = 6              # stop a model if no improvement for this many epochs
QUICK = False             # True = tiny run just to check the code works
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if QUICK:
    EPOCHS = 2
print("device:", device)
if device.type == "cpu":
    print("WARNING: no GPU found - turn on GPU T4 in Session options, CPU is far too slow")

device: cuda


In [4]:
root = "/kaggle/input/competitions/industrial-surface-defect-detection/public"
if not os.path.exists(root + "/train.csv"):
    root = "/kaggle/input/industrial-surface-defect-detection/public"

train_df = pd.read_csv(root + "/train.csv", dtype=str)
sample = pd.read_csv(root + "/sample_submission.csv", keep_default_na=False, dtype=str)
print("train:", train_df.shape, "\nsample submission:", sample.shape)

# one row per image, one column per defect (blank = no defect)
label_table = train_df.pivot(index="image_id", columns="defect_type", values="mask_rle")
label_table = label_table.reindex(columns=defects).fillna("")
print("train images:", len(label_table))
print("defects present:\n", (label_table != "").sum())

# how does the sample file write "no defect"? (blank = empty text)
print("blank mask_rle rows in sample:", (sample["mask_rle"] == "").sum(), "of", len(sample))

train: (45572, 5) 
sample submission: (12000, 4)
train images: 11393
defects present:
 defect_type
defect_a    1081
defect_b     313
defect_c    4100
defect_d    1707
dtype: int64
blank mask_rle rows in sample: 0 of 12000


In [5]:
# find every image file (train images are in the train folder)
train_dir = root + "/train"
train_files = {}
for path in glob.glob(train_dir + "/**/*.*", recursive=True):
    name = os.path.splitext(os.path.basename(path))[0]
    train_files[name] = path

ids = list(label_table.index)
if QUICK:
    ids = ids[:300]
rles = label_table.loc[ids, defects].values      # table of RLE strings

# load all images + masks into memory (uint8 keeps it small)
images = np.zeros((len(ids), HEIGHT, WIDTH), dtype=np.uint8)
masks = np.zeros((len(ids), 4, HEIGHT, WIDTH), dtype=np.uint8)

for i in range(len(ids)):
    images[i] = np.array(Image.open(train_files[ids[i]]).convert("L"))
    for k in range(4):
        rle = rles[i, k]
        if rle.strip() != "":
            numbers = np.array(rle.split(), dtype=np.int64)
            starts = numbers[0::2] - 1
            lengths = numbers[1::2]
            flat = np.zeros(HEIGHT * WIDTH, dtype=np.uint8)
            for s, l in zip(starts, lengths):
                flat[s:s + l] = 1
            masks[i, k] = flat.reshape((HEIGHT, WIDTH), order="F")   # column-by-column order

print("images:", images.shape, "| masks:", masks.shape)

images: (11393, 128, 800) | masks: (11393, 4, 128, 800)


In [6]:
# split 85% / 15%, keeping the "has any defect" balance the same in both
has_defect = masks.reshape(len(ids), -1).any(axis=1)
train_idx, val_idx = train_test_split(np.arange(len(ids)), test_size=0.15,
                                      random_state=42, stratify=has_defect)
print("train images:", len(train_idx), "| validation images:", len(val_idx))

train images: 9684 | validation images: 1709


In [7]:
models = {}

# MODEL 1 - built from scratch: a simple encoder-decoder CNN (shrink 3 times, grow 3 times)
models["scratch"] = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
    nn.Conv2d(16, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
    nn.MaxPool2d(2),                                   # 64 x 400
    nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
    nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
    nn.MaxPool2d(2),                                   # 32 x 200
    nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
    nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
    nn.MaxPool2d(2),                                   # 16 x 100
    nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
    nn.ConvTranspose2d(128, 64, 2, stride=2),          # 32 x 200
    nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
    nn.ConvTranspose2d(64, 32, 2, stride=2),           # 64 x 400
    nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
    nn.ConvTranspose2d(32, 16, 2, stride=2),           # 128 x 800
    nn.Conv2d(16, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
    nn.Conv2d(16, 4, 1),                               # 4 outputs = one mask per defect
)

# MODEL 2 - pretrained: a U-Net whose encoder is a ResNet34 already trained on ImageNet
# (needs Internet ON in the Kaggle notebook settings)
models["unet_resnet34"] = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", in_channels=1, classes=4)

# MODEL 3 - my own pick: an FPN (a different type of segmentation model) with a ResNet18 encoder
models["fpn_resnet18"] = smp.FPN(encoder_name="resnet18", encoder_weights="imagenet", in_channels=1, classes=4)

# defects are tiny, so a defect pixel counts more than a background pixel (rare defects count even more)
bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([8.0, 15.0, 3.0, 5.0]).view(1, 4, 1, 1).to(device))
for name in models:
    print(name, "parameters:", sum(p.numel() for p in models[name].parameters()))

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 46.8MB            

model.safetensors: downloading bytes:           |  0.00B            

scratch parameters: 238260
unet_resnet34 parameters: 24430532
fpn_resnet18 parameters: 13041348


In [8]:
best_scores = {}

# images that contain defect_a or defect_b (rare) get shown to the model twice per epoch
has_ab = (rles[:, 0] != "") | (rles[:, 1] != "")

for name in models:
    print("\n===== training:", name, "=====")
    model = models[name].to(device)
    lr = 1e-3 if name == "scratch" else 1e-4         # pretrained models need a gentler learning rate
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)   # halve lr when stuck
    best_score = -1
    best_weights = None
    wait = 0

    for epoch in range(EPOCHS):
        # ---------- train ----------
        model.train()
        order = np.random.permutation(np.concatenate([train_idx, train_idx[has_ab[train_idx]]]))
        total_loss = 0
        n_batches = 0
        for start in range(0, len(order), BATCH_SIZE):
            b = order[start:start + BATCH_SIZE]
            x = torch.tensor(images[b], dtype=torch.float32).unsqueeze(1) / 255.0
            y = torch.tensor(masks[b], dtype=torch.float32)

            # random flips (image and masks flipped together)
            if np.random.rand() < 0.5:
                x, y = x.flip(3), y.flip(3)
            if np.random.rand() < 0.5:
                x, y = x.flip(2), y.flip(2)
            x, y = x.to(device), y.to(device)

            if name == "scratch":
                logits = model(x)
            else:
                logits = model((x - 0.45) / 0.225)
            probs = torch.sigmoid(logits)
            inter = (probs * y).sum(dim=(2, 3))
            denom = probs.sum(dim=(2, 3)) + y.sum(dim=(2, 3))
            has_pos = (y.sum(dim=(2, 3)) > 0).float()               # 1 where the image really has that defect
            dice_loss = (1 - (2 * inter + 1) / (denom + 1)) * has_pos
            dice_loss = dice_loss.sum() / has_pos.sum().clamp(min=1)   # dice only counts on images that have the defect
            loss = bce(logits, y) + dice_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1

        # ---------- validate (threshold 0.3, score like the competition: 70% dice + 30% empty-image accuracy) ----------
        model.eval()
        dice_sum = np.zeros(4)
        pos_count = np.zeros(4)
        neg_correct = np.zeros(4)
        neg_count = np.zeros(4)
        with torch.no_grad():
            for start in range(0, len(val_idx), BATCH_SIZE):
                b = val_idx[start:start + BATCH_SIZE]
                x = torch.tensor(images[b], dtype=torch.float32).unsqueeze(1) / 255.0
                y = torch.tensor(masks[b], dtype=torch.float32)
                x = x.to(device)
                if name == "scratch":
                    logits = model(x)
                else:
                    logits = model((x - 0.45) / 0.225)
                pred = torch.sigmoid(logits).cpu() >= 0.3
                truth = y >= 0.5
                has_truth = truth.flatten(2).any(dim=2)          # (batch, 4)
                for k in range(4):
                    keep = has_truth[:, k]
                    inter = (pred[keep, k] & truth[keep, k]).flatten(1).sum(dim=1).float()
                    denom = pred[keep, k].flatten(1).sum(dim=1).float() + truth[keep, k].flatten(1).sum(dim=1).float()
                    dice_sum[k] += (2 * inter / denom).sum().item()      # one Dice per image that has the defect
                    pos_count[k] += keep.sum().item()
                    pred_any = pred[:, k].flatten(1).any(dim=1)
                    neg_correct[k] += (~pred_any[~keep]).sum().item()      # said "empty" and was right
                    neg_count[k] += (~keep).sum().item()

        fg_dice = dice_sum / np.maximum(pos_count, 1)
        empty_acc = neg_correct / np.maximum(neg_count, 1)
        score = (0.7 * fg_dice + 0.3 * empty_acc).mean()
        scheduler.step(score)
        print(f"epoch {epoch + 1}: loss {total_loss / n_batches:.4f} | val score {score:.4f} | dice per defect {fg_dice.round(3)} | empty acc {empty_acc.round(2)}", flush=True)

        # keep the best weights, stop if no improvement for a while
        if score > best_score:
            best_score = score
            best_weights = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                print("early stopping")
                break

    model.load_state_dict(best_weights)
    best_scores[name] = best_score

print("\nbest validation score of each model:")
for name in best_scores:
    print(name, round(best_scores[name], 4))

# better models get a bigger say in the final average
weights = {}
for name in best_scores:
    weights[name] = max(best_scores[name] - 0.3, 0.01)      # 0.3 = score of predicting nothing at all
total_w = sum(weights.values())
for name in weights:
    weights[name] = weights[name] / total_w
print("ensemble weights:", weights)


===== training: scratch =====
epoch 1: loss 1.0137 | val score 0.3105 | dice per defect [0.209 0.    0.428 0.456] | empty acc [0.18 0.98 0.11 0.32]
epoch 2: loss 0.7604 | val score 0.3012 | dice per defect [0.232 0.149 0.445 0.506] | empty acc [0.12 0.16 0.11 0.51]
epoch 3: loss 0.7149 | val score 0.3056 | dice per defect [0.243 0.226 0.477 0.563] | empty acc [0.11 0.14 0.07 0.23]
epoch 4: loss 0.6859 | val score 0.2846 | dice per defect [0.293 0.124 0.474 0.516] | empty acc [0.15 0.18 0.08 0.1 ]
epoch 5: loss 0.6451 | val score 0.3308 | dice per defect [0.301 0.249 0.518 0.608] | empty acc [0.06 0.07 0.04 0.33]
epoch 6: loss 0.6324 | val score 0.3468 | dice per defect [0.318 0.177 0.523 0.624] | empty acc [0.16 0.18 0.06 0.39]
epoch 7: loss 0.6242 | val score 0.3471 | dice per defect [0.322 0.219 0.531 0.625] | empty acc [0.17 0.2  0.1  0.2 ]
epoch 8: loss 0.6148 | val score 0.3613 | dice per defect [0.338 0.207 0.543 0.634] | empty acc [0.15 0.21 0.12 0.31]
epoch 9: loss 0.6088 | va

In [9]:
# predict the validation set with all 3 models and take a WEIGHTED AVERAGE of their probabilities (like blending in the tabular notebook)
for name in models:
    models[name].eval()
val_probs = np.zeros((len(val_idx), 4, HEIGHT, WIDTH), dtype=np.float16)
with torch.no_grad():
    for start in range(0, len(val_idx), BATCH_SIZE):
        b = val_idx[start:start + BATCH_SIZE]
        x = torch.tensor(images[b], dtype=torch.float32).unsqueeze(1) / 255.0
        x = x.to(device)
        total = 0
        for name in models:
            if name == "scratch":
                logits = models[name](x)
            else:
                logits = models[name]((x - 0.45) / 0.225)
            total = total + weights[name] * torch.sigmoid(logits)
        val_probs[start:start + len(b)] = total.cpu().numpy()

# for each defect, pick the best threshold AND minimum mask size, using the competition's formula:
# 70% = average Dice over images that have the defect, 30% = how often we correctly say "empty" on images that don't
candidates = np.arange(0.05, 0.951, 0.05)
min_sizes = [0, 50, 100, 200, 400, 800, 1500]
thresholds = []
min_areas = []
for k in range(4):
    truth = masks[val_idx, k].astype(bool)
    truth_area = truth.reshape(len(truth), -1).sum(axis=1)
    has_truth = truth_area > 0
    best_t, best_m, best_s = 0.5, 0, -1
    for t in candidates:
        pred = val_probs[:, k] >= t
        area = pred.reshape(len(pred), -1).sum(axis=1)
        inter_raw = (pred & truth).reshape(len(pred), -1).sum(axis=1)
        for m in min_sizes:
            keep = area >= max(m, 1)                  # masks smaller than m pixels become "no defect"
            inter = inter_raw * keep
            area_kept = area * keep

            if has_truth.sum() > 0:
                dice = (2 * inter[has_truth] / (area_kept[has_truth] + truth_area[has_truth])).mean()
            else:
                dice = 0
            if (~has_truth).sum() > 0:
                empty_acc = (~keep[~has_truth]).mean()
            else:
                empty_acc = 1

            s = 0.7 * dice + 0.3 * empty_acc
            if s > best_s:
                best_t, best_m, best_s = t, m, s
    thresholds.append(best_t)
    min_areas.append(best_m)
    print(f"{defects[k]}: threshold {best_t:.2f}, min mask size {best_m} (score {best_s:.4f})")


defect_a: threshold 0.40, min mask size 200 (score 0.5458)
defect_b: threshold 0.25, min mask size 100 (score 0.6073)
defect_c: threshold 0.55, min mask size 400 (score 0.7192)
defect_d: threshold 0.40, min mask size 400 (score 0.8014)


In [10]:
# test images (same order / ids as the sample submission)
test_ids = sample["image_id"].drop_duplicates().tolist()
test_dir = root + "/test"
if not os.path.isdir(test_dir):
    test_dir = root + "/test_images"

test_files = {}
for path in glob.glob(test_dir + "/**/*.*", recursive=True):
    name = os.path.splitext(os.path.basename(path))[0]
    test_files[name] = path
print("test images:", len(test_ids), "| found on disk:", sum(i in test_files for i in test_ids))

predictions = {}      # (image_id, defect) -> rle text
with torch.no_grad():
    for start in range(0, len(test_ids), BATCH_SIZE):
        batch_ids = test_ids[start:start + BATCH_SIZE]
        batch = np.stack([np.array(Image.open(test_files[i]).convert("L")) for i in batch_ids])
        x = torch.tensor(batch, dtype=torch.float32).unsqueeze(1) / 255.0
        x = x.to(device)
        total = 0
        for name in models:                              # weighted average of the 3 models
            if name == "scratch":
                logits = models[name](x)
            else:
                logits = models[name]((x - 0.45) / 0.225)
            total = total + weights[name] * torch.sigmoid(logits)
        probs = total.cpu().numpy()

        for n in range(len(batch_ids)):
            for k in range(4):
                mask = probs[n, k] >= thresholds[k]
                if mask.sum() < max(min_areas[k], 1):
                    mask = np.zeros_like(mask)         # too small -> treat as no defect
                # mask -> RLE text (column by column, 1-based start + length)
                pixels = np.concatenate(([0], mask.reshape(-1, order="F").astype(np.uint8), [0]))
                changes = np.flatnonzero(pixels[1:] != pixels[:-1])
                starts = changes[0::2] + 1
                lengths = changes[1::2] - changes[0::2]
                if len(starts) == 0:
                    rle = " "                    # no defect -> a single space, same as the sample file
                else:
                    rle = " ".join(f"{s} {l}" for s, l in zip(starts, lengths))
                predictions[(batch_ids[n], defects[k])] = rle
print("predictions made:", len(predictions))

test images: 3000 | found on disk: 3000
predictions made: 12000


In [11]:
import csv

submission = sample.copy()
submission["mask_rle"] = [predictions[(i, d)] for i, d in zip(submission["image_id"], submission["defect_type"])]

out_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

# empty masks are a single space " " in the sample file, and every value is in quotes
submission.to_csv(out_dir + "/submission.csv", index=False, quoting=csv.QUOTE_ALL)

# check it the way Kaggle reads it
check = pd.read_csv(out_dir + "/submission.csv", keep_default_na=False)
print("saved in:", out_dir, "->", os.listdir(out_dir))
print("rows:", len(check), "(sample has", len(sample), ") | same columns as sample:", list(check.columns) == list(sample.columns))
print("empty masks (single space):", round((check["mask_rle"] == " ").mean(), 3))
print("same IDs as sample:", (check["ID"].astype(str).values == sample["ID"].astype(str).values).all())
check.head()

saved in: /kaggle/working -> ['__notebook__.ipynb', 'submission.csv']
rows: 12000 (sample has 12000 ) | same columns as sample: True
empty masks (single space): 0.719
same IDs as sample: True


,ID,image_id,defect_type,mask_rle
0,1,WAFNKCRR7Q,defect_a,
1,2,WAFNKCRR7Q,defect_b,
2,3,WAFNKCRR7Q,defect_c,
3,4,WAFNKCRR7Q,defect_d,
4,5,YZXKEWSUDD,defect_a,
